# Notebook 07 — Score existing number rows (Gate 4B)

Assign activation-only scores; NEVER use the behavioral evaluator. Compare high/low/random deciles on held-out state reconstruction. Fast path checks scoring + leakage penalty + shuffled-subspace null.

In [1]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


FAST_DEV_RUN= False RUN_EXPENSIVE= True pkg 0.1.0


## 1. Configuration and run manifest

In [2]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="07", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


run_id: 20260717_220649_smoke_eagle_07_nogit_0


## 2. Preflight assertions

In [3]:
print('preflight: package + numpy import OK')

preflight: package + numpy import OK


## 3. Load or compute cached artifacts
aggregate score, infinite leakage penalty, and shuffled-target null destroys ranking.

In [4]:
from subliminal_icl.candidate_scoring import ScoringWeights, aggregate_score, leakage_hit_for_text
w = ScoringWeights()
clean = aggregate_score(0.9, 0.3, [0.9,0.85,0.95], 12, leakage_hit_for_text("12, 45, 3", "eagle"), w, row_id="clean")
leak = aggregate_score(0.9, 0.3, [0.9], 12, leakage_hit_for_text("eagles!", "eagle"), w, row_id="leak")
print("clean total:", round(clean.total,3), "leak total:", leak.total)
# shuffled-target subspace => random scores, no stable high decile
rng = np.random.default_rng(4)
true_scores = rng.normal(1, 0.5, 50); shuf_scores = rng.normal(0, 0.5, 50)
sep = float(np.mean(np.sort(true_scores)[-10:]) - np.mean(np.sort(shuf_scores)[-10:]))
print("high-decile separation (true - shuffled):", round(sep,3))

clean total: 1.191 leak total: -inf
high-decile separation (true - shuffled): 0.975


## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [5]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("leakage_infinite_penalty", leak.total == float("-inf"), "any semantic hit => -inf"),
          ("clean_finite", clean.total > -1e9, "clean row scored"),
          ("shuffled_null_weaker", sep > 0, "true subspace separates deciles better")]
gs = run_gate_checks("gate_04b_candidate_variance", "Candidate score variance + specificity", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


,check,result,detail
0,leakage_infinite_penalty,PASS,any semantic hit => -inf
1,clean_finite,PASS,clean row scored
2,shuffled_null_weaker,PASS,true subspace separates deciles better


GATE gate_04b_candidate_variance -> PASS


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.